# Stacking — Multi-Month Training Dataset

**Playground: seasonal_approach**

This notebook takes the feature matrix produced by `01_feature_engineering.ipynb`
and stacks it across 8 months (Apr–Nov 2025) to create the final training dataset.

**Monthly prices source: Nimit's `prices_and_months.csv`**
Rather than re-opening 8 large compressed files, we use Nimit's preprocessed file
which already has `(month, listing_id, price)` for all InsideAirbnb listings.
We filter it to our STR listing IDs from notebook 01 — this acts as the
min_nights filter since our feature matrix was already built from STR-only listings.

**Reads from (read-only outside playground):**
- `playground/seasonal_approach/data/feature_matrix.csv` — from notebook 01
- `dataset/insideairbnb nyc/nimit preprocessing/prices_and_months.csv`
- `dataset/insideairbnb nyc/2026-02 february/reviews.csv` — for seasonal index

**Writes to (playground only):**
- `playground/seasonal_approach/data/stacked_features.csv`
- `playground/seasonal_approach/data/seasonal_index.csv`

---
## 0. Setup

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

EXT     = Path('data/external')       # copied external datasets
OUT     = Path('data')
OUT.mkdir(exist_ok=True)

sources = [
    OUT / 'feature_matrix.csv',
    EXT / 'prices_and_months.csv',
    EXT / 'reviews.csv',                 # download separately — see data/external/DATASETS.md
]
for s in sources:
    print(('✓' if s.exists() else '✗ MISSING'), s.name)

✓ feature_matrix.csv
✓ prices_and_months.csv
✓ reviews.csv


---
## Step 1 — Load feature matrix

Output of `01_feature_engineering.ipynb` — 4,073 STR listings with all
spatial, structural, and host features already computed.
Price columns are stripped here; they get re-attached per month below.

In [2]:
features = pd.read_csv(OUT / 'feature_matrix.csv')

DROP = ['price_numeric', 'log_price', 'latitude', 'longitude',
        'neighbourhood_cleansed', 'neighbourhood_group_cleansed']
base = features.drop(columns=[c for c in DROP if c in features.columns])
base['id'] = base['id'].astype(str)
str_ids = set(base['id'])

feature_cols = [c for c in base.columns if c != 'id']
print(f'Feature matrix: {base.shape}')
print(f'STR listing IDs: {len(str_ids):,}')
print(f'Feature columns: {len(feature_cols)}')
assert base[feature_cols].isnull().sum().sum() == 0, 'Unexpected nulls in feature matrix'
print('No nulls ✓')

Feature matrix: (4073, 60)
STR listing IDs: 4,073
Feature columns: 59
No nulls ✓


---
## Step 2 — Load and filter Nimit's monthly prices

Nimit's file has every listing × 12 months. We filter to:
- **Months 4–11** — the only months with price data (Dec 2025+ = platform change)
- **Our STR listing IDs** — implicitly enforces min_nights < 28 since our
  feature matrix was already STR-filtered in notebook 01
- **Valid prices** — non-null, > 0 after parsing

We do NOT expand to all 44,578 listings in Nimit's file because it mixes
long-term rentals (no min_nights column available). Their November median
price is $154 vs our STR median of $222 — confirming the contamination.

In [3]:
pm = pd.read_csv(
    EXT / 'prices_and_months.csv',
    low_memory=False
)
print(f'Nimit raw shape: {pm.shape}')

pm['price_numeric'] = (
    pm['price'].astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
pm['id'] = pm['id'].astype(str)

# Price cap at $2,000/night:
# The 97th percentile is $1,595; the 98th jumps to $4,818 — a clear break.
# ~85 unique listings have $50,000 placeholder prices (hosts blocking dates
# by setting an extreme rate). Capping removes them without touching real data.
PRICE_CAP = 2000

pm_filtered = pm[
    pm['month'].between(4, 11) &
    pm['id'].isin(str_ids) &
    pm['price_numeric'].notna() &
    (pm['price_numeric'] > 0) &
    (pm['price_numeric'] <= PRICE_CAP)
][['month', 'id', 'price_numeric']].copy()

print(f'After filtering (cap=${PRICE_CAP:,}): {len(pm_filtered):,} (listing, month) pairs')
print(f'Unique listings: {pm_filtered["id"].nunique():,}')
print()
print('Rows per month:')
print(pm_filtered.groupby('month')['id'].count().to_string())

Nimit raw shape: (534936, 3)


After filtering (cap=$2,000): 28,244 (listing, month) pairs
Unique listings: 4,014

Rows per month:
month
4     3329
5     3401
6     3389
7     3439
8     3594
9     3584
10    3602
11    3906


---
## Step 3 — Derive seasonal index from review volume

COVID years (2020, 2021) excluded — see `proposed_methods.md` for full rationale.
Index is a conservative lower bound for December; review lag understates Dec demand.

In [4]:
reviews = pd.read_csv(
    EXT / 'reviews.csv',
    usecols=['listing_id', 'date'],
    parse_dates=['date']
)
reviews['year']  = reviews['date'].dt.year
reviews['month'] = reviews['date'].dt.month

clean = reviews[(reviews['year'] >= 2019) & (~reviews['year'].isin({2020, 2021}))]
print(f'Reviews used: {len(clean):,}  (years: {sorted(clean["year"].unique())})')

monthly_vol = (
    clean.groupby(['year','month']).size().reset_index(name='count')
         .groupby('month')['count'].mean().reset_index(name='avg_reviews')
)
monthly_vol['seasonal_index'] = (monthly_vol['avg_reviews'] / monthly_vol['avg_reviews'].mean()).round(4)
SEASONAL_INDEX = dict(zip(monthly_vol['month'], monthly_vol['seasonal_index']))

MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('\nSeasonal demand index:')
for m in range(1, 13):
    bar = '█' * int(SEASONAL_INDEX[m] * 20)
    print(f'  {MONTH_NAMES[m]:>3}: {SEASONAL_INDEX[m]:.4f}  {bar}')

monthly_vol.to_csv(OUT / 'seasonal_index.csv', index=False)
print(f'\nSaved seasonal_index.csv')

Reviews used: 664,458  (years: [np.int32(2019), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)])

Seasonal demand index:
  Jan: 0.7131  ██████████████
  Feb: 0.5067  ██████████
  Mar: 0.8240  ████████████████
  Apr: 0.9741  ███████████████████
  May: 1.1815  ███████████████████████
  Jun: 1.1330  ██████████████████████
  Jul: 1.1062  ██████████████████████
  Aug: 1.1690  ███████████████████████
  Sep: 1.2237  ████████████████████████
  Oct: 1.1697  ███████████████████████
  Nov: 0.9778  ███████████████████
  Dec: 1.0212  ████████████████████

Saved seasonal_index.csv


---
## Step 4 — Stack months onto feature matrix

Inner join per month: listing only gets a row for month M if it had
a valid STR price that month in Nimit's data.

In [5]:
slices = []
for month in range(4, 12):
    month_prices = pm_filtered[pm_filtered['month'] == month][['id', 'price_numeric']].copy()
    merged = base.merge(month_prices, on='id', how='inner')
    merged['month']          = month
    merged['seasonal_index'] = SEASONAL_INDEX[month]
    merged['log_price']      = np.log1p(merged['price_numeric'])
    slices.append(merged)
    print(f'Month {month:2d} ({MONTH_NAMES[month]}): {len(merged):,} rows  '
          f'median=${merged["price_numeric"].median():.0f}  '
          f'idx={SEASONAL_INDEX[month]:.3f}')

stacked = pd.concat(slices, ignore_index=True)
print(f'\nStacked: {stacked.shape}')
print(f'Unique listings: {stacked["id"].nunique():,}')
print(f'Missing values:  {stacked.isnull().sum().sum()}')

Month  4 (Apr): 3,329 rows  median=$184  idx=0.974
Month  5 (May): 3,401 rows  median=$201  idx=1.181
Month  6 (Jun): 3,389 rows  median=$200  idx=1.133
Month  7 (Jul): 3,439 rows  median=$187  idx=1.106
Month  8 (Aug): 3,594 rows  median=$189  idx=1.169
Month  9 (Sep): 3,584 rows  median=$202  idx=1.224
Month 10 (Oct): 3,602 rows  median=$207  idx=1.170
Month 11 (Nov): 3,906 rows  median=$214  idx=0.978

Stacked: (28244, 64)
Unique listings: 4,014
Missing values:  0


---
## Step 5 — Validate and save

In [6]:
NON_FEATURE = ['id', 'price_numeric', 'log_price']
model_features = [c for c in stacked.columns if c not in NON_FEATURE]

assert stacked.isnull().sum().sum() == 0
nov_median = stacked[stacked['month'] == 11]['price_numeric'].median()
assert abs(nov_median - 222) < 30, f'November median ${nov_median:.0f} unexpected'
assert 'month' in model_features
assert 'seasonal_index' in model_features
print('All assertions passed ✓')
print(f'Model input features: {len(model_features)}')
print(f'  Includes month:           {"month" in model_features}')
print(f'  Includes seasonal_index:  {"seasonal_index" in model_features}')
print(f'  New subway features:      {"subway_lines_500m" in model_features}')

out_path = OUT / 'stacked_features.csv'
stacked.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(f'Shape: {stacked.shape}')
print(f'Size:  {out_path.stat().st_size / 1e6:.1f} MB')

All assertions passed ✓
Model input features: 61
  Includes month:           True
  Includes seasonal_index:  True
  New subway features:      True



Saved: data/stacked_features.csv
Shape: (28244, 64)
Size:  8.3 MB
